# M1 · 在 W7900 上建立你自己的物理基线

**学习目标：** 在**你的** GPU 上以声明的协议产生一个真实的 Qwen-Image-2.1 B0（BF16、40 步、1024²）结果，并把已发表的 55.206 秒中位数当作*待研究的假设*，而不是你的答案。

**先预测：** 运行前先写下你预期的 warm 延迟（W7900 参考值为 55.2 秒；你的云实例可能不同——这种差异本身就是课程素材，不是错误）。

参考讲义：[M01 讲义](../course/M01_baseline/LESSON.zh-CN.md) · [English](./01_baseline_and_uncertainty.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="01")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

## 1 · 冻结参考值（永远可用，但永远不是你的）
从两个锁定的 B0 会话重算已发表的中位数——你自己的运行将与它对比。

In [ ]:
from lab import notebook_support as ns
import statistics

raw = ns.reference_warm("B0")
claimed = ns.summary_rows()["B0"]
observed = statistics.median(raw)
assert len(raw) == claimed["n_measure"] == 18
assert abs(observed - claimed["warm_total_median_s"]) < 0.03
_stamp(HISTORICAL, f"B0 reference median {observed:.3f}s "
      f"(min {min(raw):.2f}, max {max(raw):.2f}, n={len(raw)}, 2 sessions x 9)")

## 2 · 在**你的** W7900 上运行 B0
`GPU_CLASSROOM` 运行有界练习（1 个提示词 × cold+warmup+2 次测量，约 5 分钟）。`GPU_RESEARCH` 运行完整冻结协议（3 提示词 × 3 种子 × warmup+3 次测量 + fresh-prompt 臂）。两者都以只追加方式把记录写入 `runs/<run-id>/`。

In [ ]:
def run_own_b0():
    profile = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    bundle = session.run("B0", profile=profile)
    return bundle

b0 = gpu_cell(run_own_b0)
if b0 is None:
    shared = session.latest_campaign_run("B0")
    if shared:
        b0 = session.consume(shared, why="qualification-campaign B0 record")

## 3 · 你的数字，你的统计
中位数、离散度、每步时间与 VRAM 峰值——全部由上面那次运行的记录计算（REFERENCE 模式下仍来自冻结实验数据，并明确标注）。

In [ ]:
if b0 is None:
    vals, origin, steps_hint = raw, HISTORICAL, 40
    cold = None
else:
    st = b0.stats()
    vals, origin, steps_hint = b0.warm_totals(), MEASURED_TODAY, st["steps"]
    cold = b0.cold_total()

med = statistics.median(vals)
print(f"population n={len(vals)}  median={med:.3f}s  mean={statistics.mean(vals):.3f}s  "
      f"min={min(vals):.3f}s  max={max(vals):.3f}s")
if cold is not None:
    print(f"cold (first load) = {cold:.3f}s  -> warm/cold gap {cold-med:.3f}s")
_stamp(origin, f"n={len(vals)} median={med:.3f}s steps={steps_hint}")
if b0 is not None:
    print("per-step (steady median):", b0.step_stats())
    print("device VRAM peak (MB):", st["vram_device_peak_mb"], "| torch peak:",
          st["vram_peak_mb"])

## 4 · 与已发表数字对比——要解释，不要照抄
参考偏差 = 你的中位数 − 55.206 秒。几个百分比的差异属于正常的实例间波动；大差距则意味着样本总体不同（混入 cold）、资源争用（GPU 有其他租户）或温度/频率状态——请指出最可能的原因。

In [ ]:
dev = med - 55.206
print(f"reference deviation: {dev:+.3f}s ({dev/55.206*100:+.1f}% vs frozen campaign)")
print("limitation: classroom n is small; treat the deviation as a signal to "
      "investigate, not a verdict")
try:
    fig = rt.plot_ab({"label": "B0: reference vs yours",
                      "classification": "REFERENCE vs " + ("YOUR PHYSICAL RUN"
                      if b0 is not None else "REFERENCE"),
                      "baseline": {"experiment": "B0 ref", "n_warm": 18,
                                   "median_s": 55.206},
                      "candidate": {"experiment": "B0 yours", "n_warm": len(vals),
                                    "median_s": round(med, 3)}})
    if fig: fig.show()
except Exception as e:
    print("plot skipped:", e)

## 5 · 动手练习
1. 重跑一次有界练习并比较两个中位数——运行间波动比参考偏差更大还是更小？
2. 哪一个改动最能降低你的测量不确定性？为什么？

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment B0 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。